# ML 모델 모니터링

배포된 ML 모델의 성능을 모니터링하는 방법을 배웁니다.

## 학습 목표
- 모델 성능 모니터링
- 데이터 드리프트 감지
- 알림 설정
- 로깅 및 추적

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

print("모니터링 학습 시작!")

## 1. 모델 성능 모니터링

In [ ]:
# 시뮬레이션: 시간에 따른 모델 성능
np.random.seed(42)
days = 30
dates = [datetime.now() - timedelta(days=i) for i in range(days)][::-1]

# 성능 점수 시뮬레이션 (점진적 저하)
base_accuracy = 0.92
accuracy_scores = [base_accuracy - i*0.001 + np.random.randn()*0.005 for i in range(days)]

performance_df = pd.DataFrame({
    'date': dates,
    'accuracy': accuracy_scores,
    'latency_ms': np.random.uniform(50, 150, days) + np.arange(days) * 0.5
})

print(performance_df.head(10))

In [ ]:
# 성능 시각화
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# 정확도 추이
axes[0].plot(performance_df['date'], performance_df['accuracy'], 'b-o')
axes[0].axhline(y=0.90, color='r', linestyle='--', label='최소 허용 정확도')
axes[0].set_xlabel('날짜')
axes[0].set_ylabel('정확도')
axes[0].set_title('모델 정확도 추이')
axes[0].legend()
axes[0].grid(True)

# 지연 시간 추이
axes[1].plot(performance_df['date'], performance_df['latency_ms'], 'g-o')
axes[1].axhline(y=200, color='r', linestyle='--', label='최대 허용 지연시간')
axes[1].set_xlabel('날짜')
axes[1].set_ylabel('지연시간 (ms)')
axes[1].set_title('예측 지연시간 추이')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 2. 데이터 드리프트 감지

In [ ]:
# 학습 데이터 분포
np.random.seed(42)
train_data = np.random.normal(0, 1, 1000)

# 새로운 데이터 (드리프트 발생)
new_data = np.random.normal(0.5, 1.2, 500)  # 평균과 분산이 변화

plt.figure(figsize=(10, 5))
plt.hist(train_data, bins=30, alpha=0.5, label='학습 데이터', density=True)
plt.hist(new_data, bins=30, alpha=0.5, label='새로운 데이터', density=True)
plt.xlabel('값')
plt.ylabel('밀도')
plt.title('데이터 드리프트 비교')
plt.legend()
plt.show()

In [ ]:
# KS 검정으로 드리프트 감지
from scipy import stats

ks_stat, p_value = stats.ks_2samp(train_data, new_data)
print(f"KS 통계량: {ks_stat:.4f}")
print(f"p-value: {p_value:.4f}")

if p_value < 0.05:
    print("데이터 드리프트가 감지되었습니다!")
else:
    print("데이터 분포가 유사합니다.")

In [ ]:
# PSI (Population Stability Index) 계산
def calculate_psi(expected, actual, bins=10):
    """PSI 계산 함수"""
    # 구간 경계 계산
    breakpoints = np.linspace(
        min(expected.min(), actual.min()),
        max(expected.max(), actual.max()),
        bins + 1
    )
    
    # 히스토그램 계산
    expected_hist, _ = np.histogram(expected, breakpoints)
    actual_hist, _ = np.histogram(actual, breakpoints)
    
    # 비율 계산 (0 방지)
    expected_pct = (expected_hist + 1) / (len(expected) + bins)
    actual_pct = (actual_hist + 1) / (len(actual) + bins)
    
    # PSI 계산
    psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))
    return psi

psi_value = calculate_psi(train_data, new_data)
print(f"PSI: {psi_value:.4f}")

if psi_value > 0.2:
    print("심각한 드리프트! 모델 재학습이 필요합니다.")
elif psi_value > 0.1:
    print("중간 드리프트. 모니터링 강화 필요.")
else:
    print("小心翼미 드리프트.")

## 3. 예측 분포 모니터링

In [ ]:
# 예측 분포 시뮬레이션
np.random.seed(42)
n_predictions = 1000

# 시간대별 예측 분포
hours = ['00:00', '06:00', '12:00', '18:00']
predictions_by_hour = {
    '00:00': np.random.choice([0, 1], n_predictions, p=[0.7, 0.3]),
    '06:00': np.random.choice([0, 1], n_predictions, p=[0.6, 0.4]),
    '12:00': np.random.choice([0, 1], n_predictions, p=[0.5, 0.5]),
    '18:00': np.random.choice([0, 1], n_predictions, p=[0.4, 0.6])
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for idx, (hour, preds) in enumerate(predictions_by_hour.items()):
    ax = axes[idx // 2, idx % 2]
    ax.hist(preds, bins=2, edgecolor='black', alpha=0.7)
    ax.set_title(f'{hour} 예측 분포')
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['클래스 0', '클래스 1'])
    ax.set_ylabel('빈도')
    
    # 클래스 비율 표시
    ratio = preds.mean()
    ax.text(0.5, 0.9, f'클래스 1 비율: {ratio:.2%}', 
            transform=ax.transAxes, ha='center')

plt.tight_layout()
plt.show()

## 4. 알림 시스템 구현

In [ ]:
class ModelMonitor:
    def __init__(self, accuracy_threshold=0.90, latency_threshold=200, psi_threshold=0.2):
        self.accuracy_threshold = accuracy_threshold
        self.latency_threshold = latency_threshold
        self.psi_threshold = psi_threshold
        self.alerts = []
    
    def check_accuracy(self, accuracy):
        if accuracy < self.accuracy_threshold:
            alert = {
                'type': 'ACCURACY_DROP',
                'severity': 'HIGH',
                'message': f'정확도가 기준 미달: {accuracy:.4f} < {self.accuracy_threshold}',
                'timestamp': datetime.now()
            }
            self.alerts.append(alert)
            return True
        return False
    
    def check_latency(self, latency):
        if latency > self.latency_threshold:
            alert = {
                'type': 'HIGH_LATENCY',
                'severity': 'MEDIUM',
                'message': f'지연시간 초과: {latency:.1f}ms > {self.latency_threshold}ms',
                'timestamp': datetime.now()
            }
            self.alerts.append(alert)
            return True
        return False
    
    def check_drift(self, psi):
        if psi > self.psi_threshold:
            alert = {
                'type': 'DATA_DRIFT',
                'severity': 'HIGH',
                'message': f'데이터 드리프트 감지: PSI={psi:.4f} > {self.psi_threshold}',
                'timestamp': datetime.now()
            }
            self.alerts.append(alert)
            return True
        return False
    
    def get_alerts(self, severity=None):
        if severity:
            return [a for a in self.alerts if a['severity'] == severity]
        return self.alerts

# 모니터 인스턴스 생성
monitor = ModelMonitor()

# 테스트
print("알림 테스트:")
monitor.check_accuracy(0.85)  # 알림 발생
monitor.check_latency(250)    # 알림 발생
monitor.check_drift(0.25)     # 알림 발생

print(f"\n총 알림 수: {len(monitor.get_alerts())}")
for alert in monitor.get_alerts():
    print(f"  [{alert['severity']}] {alert['type']}: {alert['message']}")

## 5. 로깅 시스템

In [ ]:
import json

class PredictionLogger:
    def __init__(self, log_file='predictions.jsonl'):
        self.log_file = log_file
    
    def log_prediction(self, input_data, prediction, probability, latency_ms):
        log_entry = {
            'timestamp': datetime.now().isoformat(),
            'input': input_data,
            'prediction': prediction,
            'probability': probability,
            'latency_ms': latency_ms
        }
        
        with open(self.log_file, 'a') as f:
            f.write(json.dumps(log_entry) + '\n')
    
    def load_logs(self):
        logs = []
        with open(self.log_file, 'r') as f:
            for line in f:
                logs.append(json.loads(line))
        return pd.DataFrame(logs)

# 로깅 테스트
logger = PredictionLogger('test_predictions.jsonl')

# 샘플 예측 로깅
for i in range(5):
    logger.log_prediction(
        input_data=[float(i), float(i+1)],
        prediction=i % 2,
        probability=[0.3, 0.7],
        latency_ms=np.random.uniform(50, 150)
    )

print("로깅 테스트 완료!")

## 6. 모니터링 대시보드 요약

In [ ]:
# 모니터링 요약 리포트
def generate_monitoring_report(performance_df):
    report = """
    ====================
    모델 모니터링 리포트
    ====================
    
    1. 성능 지표
       - 현재 정확도: {accuracy:.4f}
       - 평균 지연시간: {latency:.1f}ms
       - 일일 예측 수: {predictions}
    
    2. 데이터 품질
       - 결측치 비율: {missing:.2%}
       - 이상치 비율: {outliers:.2%}
    
    3. 드리프트 상태
       - PSI: {psi:.4f}
       - 상태: {status}
    
    4. 알림
       - 미해결 알림: {alerts}건
    """
    
    return report.format(
        accuracy=performance_df['accuracy'].iloc[-1],
        latency=performance_df['latency_ms'].mean(),
        predictions=len(performance_df) * 1000,
        missing=0.001,
        outliers=0.02,
        psi=0.15,
        status='정상',
        alerts=2
    )

print(generate_monitoring_report(performance_df))

## 모니터링 체크리스트

✅ 성능 지표 모니터링 (정확도, 지연시간)
✅ 데이터 드리프트 감지
✅ 예측 분포 모니터링
✅ 알림 시스템 구축
✅ 로깅 시스템 구현
✅ 정기 리포트 생성

## 주요 모니터링 도구

| 도구 | 설명 |
|------|------|
| Prometheus | 메트릭 수집 |
| Grafana | 시각화 대시보드 |
| Evidently | ML 특화 모니터링 |
| Whylogs | 데이터 프로파일링 |
| Alibi Detect | 드리프트 감지 |